

# **Quantum Classifier - ansatz comparison**


**Authors:** Koło Naukowe Axion

**Dataset:** Banknote Authentication (UCI ML Repository)

**Framework:** Qiskit Machine Learning + PyTorch

**Abstract**

This notebook presents a comparative analysis of two different quantum variational architectures (ansatze) designed for a hybrid quantum-classical machine learning task. The primary focus is to evaluate performance differences between a standard simulator-optimized ansatz and a hardware-efficient ansatz specifically tailored for the IQM Spark (Odra) quantum computer.
Note: Default is set on 2 for depth and 0 for level of optimization. For a complex reaserch change the values: depth (2,4,6), level of optimization(0,1).


## 1. Environment Setup
This section handles dependency installation and imports. For reproducibility, all package versions should be pinned in a production environment.




### 1.1 Package Installation (Optional)

Set `INSTALL_DEPS = True` if running in a fresh environment. For production use, pin specific versions.

In [ ]:
# @title
# Optional: Install dependencies if not already present
INSTALL_DEPS = True

if INSTALL_DEPS:
    import sys
    import subprocess

    packages = [
        'numpy',
        'scikit-learn',
        'kagglehub[pandas-datasets]',
        'qiskit',
        'qiskit_algorithms',
        'qiskit-machine-learning',
        'iqm-client[qiskit]',
        'torch',
        'matplotlib',
        'pandas'
    ]

    for pkg in packages:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

### 1.2 Imports


In [ ]:
# Standard library
import random
from pathlib import Path

# Scientific computing
import numpy as np
import matplotlib.pyplot as plt
import csv
import pandas as pd
import kagglehub

# Machine learning
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, f1_score, accuracy_score
from IPython.display import display

from qiskit import QuantumCircuit
from qiskit import transpile
from qiskit.circuit import ParameterVector
from qiskit.primitives import StatevectorEstimator, PrimitiveResult, PubResult
from qiskit.primitives.base import BaseEstimatorV2
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives.containers.data_bin import DataBin
from qiskit_machine_learning.connectors import TorchConnector
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_algorithms.gradients import ParamShiftEstimatorGradient

#Third-party: Setting up Quantum Hardware
from iqm.qiskit_iqm import IQMProvider

## 2. Reproducibility and Random Seed Control

In [ ]:
def set_random_seed(seed: int = 42) -> None:
    """
    Set random seeds for reproducibility across numpy, PyTorch, and Python's random module.

    Parameters
    ----------
    seed : int
        Random seed value (default: 42)
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# Set global seed
RANDOM_SEED = 42
set_random_seed(RANDOM_SEED)

## 3. Data Preparation

The **Banknote Authentication Dataset** contains features extracted from images of genuine and forged banknotes. We perform feature engineering (interaction term) and scale features to the range [0, π] for angle encoding.

In [ ]:
def prepare_data(test_size: float = 0.2, random_state: int = 42):
    # Download the Kaggle dataset locally and read the CSV file from the cached folder.
    dataset_dir = Path(kagglehub.dataset_download("shanks0465/banknoteauthentication"))
    csv_files = sorted(dataset_dir.glob("*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"No CSV file found in Kaggle dataset folder: {dataset_dir}")

    dataset = pd.read_csv(csv_files[0])
    dataset.columns = [column.strip().lower() for column in dataset.columns]

    feature_columns = dataset.columns[:-1]
    target_column = dataset.columns[-1]

    X = dataset[feature_columns].to_numpy(dtype=float)
    y = dataset[target_column].to_numpy().ravel()

    indices = np.arange(len(X))

    # Sanity checks
    assert X.shape[1] == 4, f"Expected 4 features, got {X.shape[1]}"
    assert set(np.unique(y)) == {0, 1}, f"Expected binary labels {{0, 1}}, got {set(np.unique(y))}"

    # Feature engineering: interaction term
    variance = X[:, 0].reshape(-1, 1)
    skewness = X[:, 1].reshape(-1, 1)
    interaction = variance * skewness
    X_expanded = np.hstack((X, interaction))

    X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
        X_expanded, y, indices, test_size=test_size, random_state=random_state
    )

    # Scale features to [-pi/4, pi/4] for angle encoding
    scaler = MinMaxScaler(feature_range=(-np.pi/4, np.pi/4))
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    print(f"Dataset loaded from Kaggle: {csv_files[0].name}")
    print(f"Dataset loaded: {len(X_train)} train, {len(X_test)} test samples")
    print(f"Feature dimension: {X_train_scaled.shape[1]}")

    return X_train_scaled, X_test_scaled, y_train, y_test, idx_train, idx_test

# 4. Ansatz 1: Ring Topology (Simulation-Optimized)

In [ ]:
def ansatz_trimmed_reverse_q0_param_count(n_qubits: int, depth: int) -> int:
    """Weights when only the last macro-layer uses the q0-incident reverse trim."""
    n_macro = depth // 2
    if n_macro == 0:
        return 0
    full_layer = 4 * n_qubits
    last_layer = 3 * n_qubits + 2
    return (n_macro - 1) * full_layer + last_layer


def ansatz(n_qubits: int, depth: int) -> QuantumCircuit:
    n_macro = depth // 2
    theta = ParameterVector("theta", ansatz_trimmed_reverse_q0_param_count(n_qubits, depth))
    qc = QuantumCircuit(n_qubits)
    param_idx = 0

    for j in range(n_macro):
        last_layer = j == n_macro - 1

        for i in range(n_qubits):
            qc.ry(theta[param_idx], i)
            param_idx += 1

        for i in range(n_qubits):
            control = i
            target = (i + 1) % n_qubits
            qc.crx(theta[param_idx], control, target)
            param_idx += 1

        for i in range(n_qubits):
            qc.rx(theta[param_idx], i)
            param_idx += 1

        if last_layer:
            for k in range(2):
                i = k
                control = i
                target = (i - 1) % n_qubits
                qc.cry(theta[param_idx], control, target)
                param_idx += 1
        else:
            for i in range(n_qubits):
                control = i
                target = (i - 1) % n_qubits
                qc.cry(theta[param_idx], control, target)
                param_idx += 1

    assert param_idx == len(theta)
    return qc

In [ ]:
ma=ansatz(5,2)
ma.draw(style="mpl")

### 4.2 Hybrid Variational Quantum Circuit

The `HybridModel` class implements a VQC by integrating the quantum circuit with PyTorch's autograd system via Qiskit's `TorchConnector`. This enables gradient-based optimization of quantum parameters using classical optimizers.

In [ ]:
class HybridModel(nn.Module):
    """
    Hybrid Variational Quantum Circuit (VQC) for binary classification.

    The model combines:
    1. Angle encoding feature map (classical data → quantum state)
    2. Parametrized ansatz (trainable quantum circuit)
    3. Observable measurement (quantum state → classical expectation value)
    4. PyTorch integration via TorchConnector (enables backpropagation)

    Parameters
    ----------
    ansatz_circuit : QuantumCircuit
        Parametrized quantum circuit with trainable weights
    num_qubits : int
        Number of qubits (must match feature dimension)

    Attributes
    ----------
    qnn : EstimatorQNN
        Qiskit's EstimatorQNN that computes expectation values
    quantum_layer : TorchConnector
        PyTorch-compatible wrapper enabling gradient computation

    Notes
    -----
    - **Feature encoding**: RY(x_i) on qubit i encodes feature x_i
    - **Observable**: Pauli-Z on qubit 0, measuring spin in computational basis
    - **Output range**: [-1, +1] (expectation value of Z operator)
    - **Gradient method**: Reverse Estimator Gradient quantum gradients
    - **Simulator**: StatevectorEstimator (change for real quantum hardware)
    """

    def __init__(self, ansatz_circuit, num_qubits):
        super().__init__()
        # Create angle encoding feature map
        self.feature_map = self._create_angle_encoding(num_qubits)

        # Compose full quantum circuit: feature_map → ansatz
        self.qc = QuantumCircuit(num_qubits)
        self.qc.compose(self.feature_map, qubits=range(num_qubits), inplace=True)
        self.qc.compose(ansatz_circuit, inplace=True)

        # Separate input parameters (from feature map) and weight parameters (from ansatz)
        # This distinction is crucial for EstimatorQNN to correctly handle data vs. trainable weights
        input_params = list(self.feature_map.parameters)
        weight_params = list(ansatz_circuit.parameters)

        # Define observable: measure Z on qubit 0 (identity on other qubits)
        # Pauli string ordering: rightmost character = qubit 0
        # Example for 5 qubits: "IIIIZ" measures Z on q0, I on q1-q4
        observable = SparsePauliOp.from_list([("I" * (num_qubits - 1) + "Z", 1)])

        # Initialize statevector simulator for noiseless quantum simulation
        # NOTE: Replace with Sampler or real backend for quantum hardware deployment
        estimator = StatevectorEstimator()

        # Use parameter shift rule for computing quantum gradients
        # This is exact (not finite-difference) and works on hardware
        gradient = ParamShiftEstimatorGradient(estimator)

        # Create variational quantum circuit using EstimatorQNN
        # EstimatorQNN computes <ψ|O|ψ> where |ψ> = ansatz(weights)|feature_map(x)>
        self.qnn = EstimatorQNN(
            circuit=self.qc,
            observables=observable,
            input_params=input_params,
            weight_params=weight_params,
            estimator=estimator,
            gradient=gradient
        )
        # Wrap the VQC as a PyTorch module
        # TorchConnector bridges Qiskit and PyTorch autograd systems,
        # allowing standard PyTorch optimizers (SGD, Adam, etc.) to train quantum parameters
        self.quantum_layer = TorchConnector(self.qnn)

    def _create_angle_encoding(self, num_qubits: int) -> QuantumCircuit:
        """
        Create angle encoding feature map: |0⟩ → RY(x₀) ⊗ RY(x₁) ⊗ ... ⊗ RY(xₙ) |0⟩

        Each classical feature x_i ∈ [0, π] is encoded as a rotation angle on qubit i.
        This maps the feature vector to the amplitude of the quantum state.

        Parameters
        ----------
        num_qubits : int
            Number of qubits (and features)

        Returns
        -------
        QuantumCircuit
            Feature map circuit with n_qubits input parameters
        """
        qc_data = QuantumCircuit(num_qubits)
        input_params = ParameterVector('x', num_qubits)
        for i in range(num_qubits):
            qc_data.ry(input_params[i], i)
        return qc_data

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass through the hybrid variational quantum circuit.

        Parameters
        ----------
        x : torch.Tensor
            Input features, shape (batch_size, num_qubits)
         Returns
        -------
        torch.Tensor
            Expectation values, shape (batch_size, 1), range [-1, +1]
        """
        return self.quantum_layer(x)

## 5. Training Configuration and Data Loading

In [ ]:
EPOCHS = 30
BATCH_SIZE = 16
LEARNING_RATE = 0.01
NUM_QUBITS = 5
ANSATZ_DEPTH = 2

X_train, X_test, y_train_raw, y_test_raw, train_idx, test_idx = prepare_data(
    test_size=0.2,
    random_state=RANDOM_SEED
)

# --- SAVE TEST ---
with open('test_set_indices.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['sample_index', 'random_seed'])
    for idx in test_idx.tolist():
        writer.writerow([idx, RANDOM_SEED])

# --- SAVE TRAIN ---
with open('train_set_indices.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['sample_index', 'random_seed'])
    for idx in train_idx.tolist():
        writer.writerow([idx, RANDOM_SEED])

print(f"Indices saved: 'test_set_indices.csv' and 'train_set_indices.csv'")
# --------------------

# Map labels from {0, 1} to {-1, +1}
y_train = 2 * y_train_raw - 1
y_test = 2 * y_test_raw - 1

# Convert to float32
X_train = X_train.astype(np.float32)
X_test = X_test.astype(np.float32)
y_train = y_train.astype(np.float32)
y_test = y_test.astype(np.float32)

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)

In [ ]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)

# Create DataLoader for batched training
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

## 6. Model Training, Evaluation & Weights upload

We train the hybrid VQC using Mean Squared Error (MSE) loss and the Adam optimizer. The training loop evaluates both training and test performance at each epoch.

### 6.1. Training



In [ ]:
def train_epoch(model, train_loader, optimizer, loss_fn):
    """
    Train the VQC model for one epoch.

    Parameters
    ----------
    model : nn.Module
        Hybrid variational quantum circuit model
    train_loader : DataLoader
        Training data loader
    optimizer : torch.optim.Optimizer
        Optimizer for updating model parameters
    loss_fn : nn.Module
        Loss function (e.g., MSELoss)

    Returns
    -------
    float
        Average training loss over all batches
    """
    model.train()
    epoch_loss = 0.0
    num_batches = 0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        output = model(X_batch)
        loss = loss_fn(output, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        num_batches += 1

    return epoch_loss / num_batches


def evaluate(model, X_tensor, y_tensor, loss_fn):
    """
    Evaluate the VQC model on a dataset.

    Parameters
    ----------
    model : nn.Module
        Hybrid variational quantum circuit model
    X_tensor : torch.Tensor
        Input features
    y_tensor : torch.Tensor
        True labels (in {-1, +1})
    loss_fn : nn.Module
        Loss function

    Returns
    -------
    dict
        Dictionary containing 'loss' and 'accuracy'
    """
    model.eval()
    with torch.no_grad():
        outputs = model(X_tensor)
        loss = loss_fn(outputs, y_tensor).item()

        # Convert continuous outputs to binary predictions
        # outputs > 0 → class +1, outputs ≤ 0 → class -1
        predicted = (outputs > 0).float() * 2 - 1
        correct = (predicted == y_tensor).sum().item()
        accuracy = correct / len(y_tensor)

    return {'loss': loss, 'accuracy': accuracy}

# Initialize model, loss, and optimizer
original_ansatz = ansatz(NUM_QUBITS, ANSATZ_DEPTH)
original_model = HybridModel(original_ansatz, NUM_QUBITS)
original_loss_function = nn.MSELoss()
original_optimizer = torch.optim.Adam(original_model.parameters(), lr=LEARNING_RATE)



In [ ]:
# Training loop
original_train_loss_history = []
original_test_loss_history = []
original_test_acc_history = []

print(f"Starting training for {EPOCHS} epochs...")
print(f"Original model: {NUM_QUBITS} qubits, ansatz depth {ANSATZ_DEPTH}")
print(f"Trainable parameters: {sum(p.numel() for p in original_model.parameters() if p.requires_grad)}")
print("-" * 60)

original_best_accuracy = 0.0

for epoch in range(EPOCHS):
    train_loss = train_epoch(original_model, train_loader, original_optimizer, original_loss_function)
    test_metrics = evaluate(original_model, X_test_tensor, y_test_tensor, original_loss_function)

    current_acc = test_metrics['accuracy']
    if current_acc > original_best_accuracy:
        original_best_accuracy = current_acc
        torch.save(original_model.state_dict(), "quantum_best_weights_2.pth")
        print(f" New best accuracy: {original_best_accuracy:.4f} (Original weights updated)")

    original_train_loss_history.append(train_loss)
    original_test_loss_history.append(test_metrics['loss'])
    original_test_acc_history.append(test_metrics['accuracy'])

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{EPOCHS} | "
              f"Train Loss: {train_loss:.4f} | "
              f"Test Acc: {test_metrics['accuracy']:.4f}")

torch.save(original_model.state_dict(), "quantum_final_weights_2.pth")
print("-" * 60)
print(f"Training complete! Best original accuracy achieved: {original_best_accuracy:.4f}")

### 6.2. Evaluation


In [ ]:
with torch.no_grad():
    test_outputs_tensor = original_model(X_test_tensor)
    test_outputs = test_outputs_tensor.numpy()

predicted = np.where(test_outputs > 0, 1, -1).flatten()

c_matrix_display = ConfusionMatrixDisplay(confusion_matrix=confusion_matrix(y_test, predicted))
c_matrix_display.plot()

epochs = range(1, len(original_train_loss_history) + 1)

plt.figure(figsize=(14, 5))

# Plot 1: Loss
plt.subplot(1, 2, 1)
plt.plot(epochs, original_train_loss_history, label='Train Loss', color='blue')
plt.plot(epochs, original_test_loss_history, label='Test Loss', color='red', linestyle='--')
plt.title('Original Ansatz - Loss Over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Plot 2: Accuracy
plt.subplot(1, 2, 2)
plt.plot(epochs, original_test_acc_history, label='Test Accuracy', color='green')
plt.title('Original Ansatz - Accuracy Over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.show()

print("F1 SCORE: ", f1_score(y_test, predicted), " | ACCURACY SCORE: ", accuracy_score(y_test, predicted), " | BEST ACCURACY SCORE: ", max(original_test_acc_history))

### 6.3. Weights Upload

To facilitate the immediate verification of our results, this cell provides an option to load the optimal model parameters `quantum_best_weights.pth`. This allows users to skip the computationally intensive training phase and proceed directly to inference and hardware-level validation using our pre-trained state.

In [ ]:
# Use to load weights from the file, otherwise skip
final_ansatz = ansatz(NUM_QUBITS, ANSATZ_DEPTH)
original_model = HybridModel(final_ansatz, NUM_QUBITS)
FILE_PATH = "quantum_final_weights_2.pth"
try:
    original_model.load_state_dict(torch.load(FILE_PATH))
    original_model.eval()
    sanity_metrics = evaluate(original_model, X_test_tensor, y_test_tensor, nn.MSELoss())
    print(f"Weights loaded successfully from {FILE_PATH}! You can now proceed to hardware testing.")
    print(f"Estimator sanity accuracy: {sanity_metrics['accuracy']:.4f}")
except FileNotFoundError:
    print("Error: The weights file was not found.")

## 7. Hardware Setup

### 7.1. Custom IQM Hardware Execution & Expectation Calculation

This block implements the core execution loop for a custom Qiskit Estimator primitive, designed to interface specifically with IQM Spark ODRA 5 quantum hardware. It takes parameterized circuits (PUBs), binds them to specific values, runs them on the backend, and calculates the expectation value.



**Step-by-Step Workflow:**
1. **Parameter Binding:** Extracts the `parameter_values` from each PUB and binds them to the pre-compiled (transpiled) quantum circuit.
2. **Hardware Execution:** Submits the bound circuit to the IQM backend (`self._backend.run`) for a specified number of `SHOTS` and retrieves the measurement counts.
3. **Expectation Calculation:** Calculates the expectation value for the Pauli-Z operator ($\langle Z \rangle$). It calculates the probability of measuring a 0 ($p_0$) and uses the formula $p_0 - p_1$ to map the result to a range between -1 and +1.
4. **Result Formatting:** Packages the final expectation values into Qiskit V2 primitive objects (`DataBin`, `PubResult`) and returns a `SimpleIQMJob`.

**Important Implementation Notes:**

**Hardcoded Target:** Due to Qiskit's bitstring formatting, the calculation `bitstring[-1] == '0'` explicitly targets **Qubit 0** (the very first qubit), ignoring the states of all other qubits.

In [ ]:
class SimpleIQMJob:
    """A dummy job that simply holds the result."""
    def __init__(self, result):
        self._result = result

    def result(self):
        return self._result

# --- THE BRIDGE CLASS ---
class IQMBackendEstimator(BaseEstimatorV2):
    def __init__(self, backend, options=None):
        super().__init__()
        self._backend = backend
        self._options = options or {"shots": 100}
        # collecting timestamps
        self.timestamp_history = []
        self.total_qpu_time = 0.0  # Sum of time on quantum

    def _extract_timestamps(self, result):
        try:
            timeline = result._metadata.get('timeline', [])
            if not timeline:
                return None

            timestamps = {}
            for entry in timeline:
                timestamps[entry.status] = entry.timestamp

            return timestamps
        except Exception:
            return None
    '''

    def run(self, pubs, precision=None):
        if not isinstance(pubs, list): pubs = [pubs]
        job_results = []

        # 1. Prepare Circuit
        base_circuit = pubs[0][0]
        circuit_with_meas = base_circuit.copy()
        if circuit_with_meas.num_clbits == 0:
            circuit_with_meas.measure_all()

        # 2. Transpile
        transpiled_qc = transpile(circuit_with_meas, self._backend, optimization_level=0)

        for pub in pubs:
            _, observables, parameter_values = pub
            if parameter_values.ndim == 1:
                parameter_values = [parameter_values]

            pub_expectations = []

            for params in parameter_values:
                bound_qc = transpiled_qc.assign_parameters(params)

                # 3. Execute on Hardware
                try:
                    job = self._backend.run(bound_qc, shots=self._options["shots"])
                    result = job.result()

                    # ========== TIMESTAMPS (IQM timeline) ==========
                    ts = self._extract_timestamps(result)
                    if ts:
                        exec_start = ts.get('execution_started')
                        exec_end = ts.get('execution_ended')
                        comp_start = ts.get('compilation_started')
                        comp_end = ts.get('compilation_ended')
                        job_created = ts.get('created')
                        job_completed = ts.get('completed')

                        if exec_start and exec_end:
                            execution_time = (exec_end - exec_start).total_seconds()
                            compile_time = (comp_end - comp_start).total_seconds() if comp_start and comp_end else 0
                            job_time = (job_completed - job_created).total_seconds() if job_created and job_completed else 0
                            self.timestamp_history.append({
                                'execution_time_qpu': execution_time,
                                'job_time_total': job_time,
                                'compile_time': compile_time,
                                'raw_timestamps': ts
                            })
                            self.total_qpu_time += execution_time

                            print(f"TIME ON QPU: {execution_time*1000:.2f}ms | "
                                  f"Compilation: {compile_time*1000:.2f}ms | "
                                  f"Job overall: {job_time:.3f}s")
                    # =========================================================

                    counts = result.get_counts()

                    if isinstance(counts, list): counts = counts[0]

                    # 4. Calculate Expectation
                    shots = sum(counts.values())
                    count_0 = 0
                    for bitstring, count in counts.items():
                        if bitstring[-1] == '0':
                            count_0 += count

                    p0 = count_0 / shots
                    p1 = 1 - p0
                    pub_expectations.append(p0 - p1)

                except Exception as e:
                    print(f"Job failed: {e}")
                    pub_expectations.append(0.0)

            data = DataBin(evs=np.array(pub_expectations), shape=(len(pub_expectations),))
            job_results.append(PubResult(data=data))

        return SimpleIQMJob(PrimitiveResult(job_results))'''
    
    def _counts_to_expectation(self, counts):
        if isinstance(counts, list):
            counts = counts[0]
        shots = sum(counts.values())
        count_0 = sum(count for bitstring, count in counts.items() if bitstring[-1] == '0')
        p0 = count_0 / shots if shots else 0
        return p0 - (1 - p0)

    def run(self, pubs, precision=None):
        if not isinstance(pubs, list):
            pubs = [pubs]
        job_results = []
        shots_opt = self._options["shots"]
        max_circuits = self._options.get("max_circuits_per_job")

        # 1. Prepare and transpile base circuit once
        base_circuit = pubs[0][0]
        circuit_with_meas = base_circuit.copy()
        if circuit_with_meas.num_clbits == 0:
            circuit_with_meas.measure_all()
        transpiled_qc = transpile(circuit_with_meas, self._backend, optimization_level=0)

        for pub in pubs:
            _, observables, parameter_values = pub
            if parameter_values.ndim == 1:
                parameter_values = [parameter_values]

            # 2. Build all bound circuits first
            bound_circuits = [
                transpiled_qc.assign_parameters(params)
                for params in parameter_values
            ]
            n_circuits = len(bound_circuits)

            # 3. Execute in batches
            pub_expectations = []
            for start in range(0, n_circuits, max_circuits or n_circuits):
                end = min(start + (max_circuits or n_circuits), n_circuits)
                batch = bound_circuits[start:end]

                try:
                    job = self._backend.run(batch, shots=shots_opt)
                    result = job.result()

                    ts = self._extract_timestamps(result)
                    if ts:
                        exec_start = ts.get('execution_started')
                        exec_end = ts.get('execution_ended')
                        comp_start = ts.get('compilation_started')
                        comp_end = ts.get('compilation_ended')
                        job_created = ts.get('created')
                        job_completed = ts.get('completed')

                        if exec_start and exec_end:
                            execution_time = (exec_end - exec_start).total_seconds()
                            compile_time = (comp_end - comp_start).total_seconds() if comp_start and comp_end else 0
                            job_time = (job_completed - job_created).total_seconds() if job_created and job_completed else 0
                            self.timestamp_history.append({
                                'execution_time_qpu': execution_time,
                                'job_time_total': job_time,
                                'compile_time': compile_time,
                                'raw_timestamps': ts,
                                'n_circuits': len(batch),
                            })
                            self.total_qpu_time += execution_time
                            print(
                                f"Batch of {len(batch)} circuits | QPU: {execution_time*1000:.0f}ms | "
                                f"Compile: {compile_time*1000:.0f}ms | Job: {job_time:.3f}s"
                            )

                    counts_list = result.get_counts()
                    if not isinstance(counts_list, list):
                        counts_list = [counts_list]

                    for counts in counts_list:
                        pub_expectations.append(self._counts_to_expectation(counts))

                except Exception as e:
                    print(f"Batch job failed: {e}")
                    pub_expectations.extend([0.0] * len(batch))

            data = DataBin(evs=np.array(pub_expectations), shape=(len(pub_expectations),))
            job_results.append(PubResult(data=data))

        return SimpleIQMJob(PrimitiveResult(job_results))
    

    def print_timing_summary(self):
        """Detailed summary."""
        if not self.timestamp_history:
            print("Brak danych o timestampach.")
            return

        print("\n" + "="*60)
        print("DETAILED SUMMARY OF THE TIMESTAMPS")
        print("="*60)
        print(f"Number of executed jobs: {len(self.timestamp_history)}")

        qpu_times = []
        compile_times = []
        queue_times = []
        network_times = []

        for t in self.timestamp_history:
            ts = t['raw_timestamps']

            # QPU
            if ts.get('execution_started') and ts.get('execution_ended'):
                qpu_times.append((ts['execution_ended'] - ts['execution_started']).total_seconds())

            # Compilation
            if ts.get('compilation_started') and ts.get('compilation_ended'):
                compile_times.append((ts['compilation_ended'] - ts['compilation_started']).total_seconds())

            # Queue (waiting for QPU)
            if ts.get('pending_execution') and ts.get('execution_started'):
                queue_times.append((ts['execution_started'] - ts['pending_execution']).total_seconds())

            # (created->received + ready->completed)
            net_time = 0
            if ts.get('created') and ts.get('received'):
                net_time += (ts['received'] - ts['created']).total_seconds()
            if ts.get('ready') and ts.get('completed'):
                net_time += (ts['completed'] - ts['ready']).total_seconds()
            network_times.append(net_time)

        print(f"\nTIME ON QPU :     {sum(qpu_times)*1000:8.2f} ms  (mean: {np.mean(qpu_times)*1000:.2f} ms/job)")
        print(f"Compilation :           {sum(compile_times)*1000:8.2f} ms  (mean: {np.mean(compile_times)*1000:.2f} ms/job)")
        print(f"Queue (wait QPU) :   {sum(queue_times)*1000:8.2f} ms  (mean: {np.mean(queue_times)*1000:.2f} ms/job)")
        print(f"(upload+down) :   {sum(network_times)*1000:8.2f} ms  (mean: {np.mean(network_times)*1000:.2f} ms/job)")

        total_measured = sum(qpu_times) + sum(compile_times) + sum(queue_times) + sum(network_times)
        total_job = sum(t['job_time_total'] for t in self.timestamp_history)
        other = total_job - total_measured

        print(f"Others (validation etc): {other*1000:8.2f} ms")
        print(f"\nTIME OVERALL:       {total_job*1000:8.2f} ms ({total_job:.3f} s)")

        print("\n" + "-"*40)
        print("PERCENTAGE DISTRIBUTION: ")
        print(f"  QPU:        {100*sum(qpu_times)/total_job:5.1f}%")
        print(f"  Compilation: {100*sum(compile_times)/total_job:5.1f}%")
        print(f"  Queue:    {100*sum(queue_times)/total_job:5.1f}%")
        print(f"  Network:       {100*sum(network_times)/total_job:5.1f}%")
        print(f"  Others:       {100*other/total_job:5.1f}%")
        print("="*60 + "\n")


### 7.2. Constructing the Hardware-Ready Quantum Neural Network (QNN)

This section builds the bridge between PyTorch machine learning environment and the physical IQM quantum hardware. We construct a new quantum neural network explicitly bound to the hardware estimator, and then transfer our pre-trained weights into it for real-world inference.



**Key Steps in this Cell:**
* **Hardware Estimator Setup:** Initializes the custom `IQMBackendEstimator` to handle execution and shot-counting on the physical backend.
* **Circuit Assembly:** Constructs the physical quantum blueprint (`hw_qc`) by combining our data-encoding Feature Map and our trainable Ansatz.
* **Defining the Observable:** Sets up a `SparsePauliOp` (`IIIIZ`) to measure the Pauli-Z expectation value of Qubit 0, aligning with our estimator's logic.
* **PyTorch Integration (`TorchConnector`):** Wraps the Qiskit `EstimatorQNN` into a PyTorch module (`iqm_model`), allowing it to natively accept PyTorch tensors as input.
* **Weight Transfer & Evaluation Mode:** Extracts the optimized weights from the classically-trained model and injects them into our new hardware-linked model. Calling `.eval()` locks the model into testing mode, ensuring we don't waste hardware resources calculating gradients.

In [ ]:
# Connect to IQM
# ---------------------------------------------------------
try:
    # Replace URL/Token as needed
    provider = IQMProvider("https://odra5.e-science.pl/", token=input("Enter IQM Token: "))
    iqm_backend = provider.get_backend()
    print(f"Connected to backend: {iqm_backend.name}")
except Exception as e:
    print(f"Connection error: {e}")
    # Stop execution if connection fails (optional safety)
    exit()

# Instantiate the Bridge & QNN
# ---------------------------------------------------------

# Create the hardware estimator
ORIGINAL_SHOTS = 512
SHOTS = 512
MAX_CIRCUITS_PER_JOB = 275

original_hardware_estimator = IQMBackendEstimator(
    iqm_backend,
    options={
        "shots": SHOTS,
        "max_circuits_per_job": MAX_CIRCUITS_PER_JOB,
    }
)

print("Building Original Ansatz Hardware QNN...")

# Build Circuit Components
original_hw_ansatz = ansatz(NUM_QUBITS, ANSATZ_DEPTH)
original_hw_feature_map = original_model._create_angle_encoding(NUM_QUBITS)

original_hw_qc = QuantumCircuit(NUM_QUBITS)
original_hw_qc.compose(original_hw_feature_map, qubits=range(NUM_QUBITS), inplace=True)
original_hw_qc.compose(original_hw_ansatz, inplace=True)

# Observable
observable = SparsePauliOp.from_list([("I" * (NUM_QUBITS - 1) + "Z", 1)])

# Create QNN with the HARDWARE ESTIMATOR
original_hw_qnn = EstimatorQNN(
    circuit=original_hw_qc,
    observables=observable,
    input_params=list(original_hw_feature_map.parameters),
    weight_params=list(original_hw_ansatz.parameters),
    estimator=original_hardware_estimator
)

# Create Torch Layer
original_iqm_model = TorchConnector(original_hw_qnn)

# Load trained weights
# ---------------------------------------------------------
print("Loading original weights...")
try:
    original_iqm_model.load_state_dict(original_model.quantum_layer.state_dict())
    original_iqm_model.eval()
    print("Original ansatz weights transferred to IQM model successfully!")
except RuntimeError as e:
    print("\n[!] Weight Loading Error: The shape of the trained weights does not match the hardware model.")
    print("Ensure the model you are loading from was trained with N_QUBITS=5 and the exact same original ansatz.")
    print(f"Details: {e}")

## 8. Testing Single Input on IQM Spark

This section sends a randomly chosen sample to IQM Spark to test our model on real quantum hardware.

In [ ]:
def count_gate_types(circuit):
    total_gates = 0
    single_qubit_gates = 0
    two_qubit_gates = 0
    single_qubit_gate_counts = {qubit: 0 for qubit in range(circuit.num_qubits)}
    two_qubit_gate_counts = {
        (q1, q2): 0
        for q1 in range(circuit.num_qubits)
        for q2 in range(q1 + 1, circuit.num_qubits)
    }

    for instruction in circuit.data:
        operation = instruction.operation if hasattr(instruction, "operation") else instruction[0]
        qubits = instruction.qubits if hasattr(instruction, "qubits") else instruction[1]

        if operation.name in {"barrier", "measure"}:
            continue

        qubit_indices = tuple(circuit.find_bit(qubit).index for qubit in qubits)
        total_gates += 1

        if operation.num_qubits == 1:
            single_qubit_gates += 1
            single_qubit_gate_counts[qubit_indices[0]] += 1
        elif operation.num_qubits == 2:
            two_qubit_gates += 1
            pair = tuple(sorted(qubit_indices))
            two_qubit_gate_counts[pair] += 1

    return {
        'Total Gates': total_gates,
        'Single-Qubit Gates': single_qubit_gates,
        'Two-Qubit Gates': two_qubit_gates,
        'Single-Qubit Gates by Qubit': single_qubit_gate_counts,
        'Two-Qubit Gates by Pair': two_qubit_gate_counts,
    }


def print_gate_breakdown(stats):
    print(f"Total Gates:        {stats['Total Gates']}")
    print(f"Single-Qubit Gates: {stats['Single-Qubit Gates']}")
    print(f"Two-Qubit Gates:    {stats['Two-Qubit Gates']}")
    print("Single-Qubit Gates on Each Qubit:")
    for qubit, count in stats['Single-Qubit Gates by Qubit'].items():
        print(f"  q{qubit}: {count}")
    print("Two-Qubit Gates by Pair:")
    for (q1, q2), count in stats['Two-Qubit Gates by Pair'].items():
        print(f"  q{q1}-q{q2}: {count}")


def get_circuit_stats(circuit, backend):
    # Transpile to see the circuit in the backend gate set used by the quantum computer.
    t_qc = transpile(circuit, backend, optimization_level=0)
    ops = t_qc.count_ops()
    stats = {
        'Depth': t_qc.depth(),
        'SWAPs': ops.get('swap', 0),
    }
    stats.update(count_gate_types(t_qc))
    return t_qc, stats

In [ ]:
qnn_circuit = original_iqm_model.neural_network.circuit
transpiled_qnn_circuit, stats = get_circuit_stats(qnn_circuit, iqm_backend)

hardware_test_inputs = X_test_tensor
hardware_test_labels = y_test_tensor.view(-1)
num_samples_to_test = len(hardware_test_inputs)

circuits_per_job = original_hardware_estimator._options.get("max_circuits_per_job") or num_samples_to_test
num_jobs = int(np.ceil(num_samples_to_test / circuits_per_job))

print(f"Loaded ALL {num_samples_to_test} test samples for hardware testing.")
print(f"\nSending {num_jobs} job(s) containing up to {circuits_per_job} batched circuits to IQM Spark...")
print("\nTranspiled circuit used on the quantum computer:")
display(transpiled_qnn_circuit.draw(style="mpl", idle_wires=False))

with torch.no_grad():
    predictions = original_iqm_model(hardware_test_inputs).view(-1)

predicted_labels = torch.where(predictions > 0, 1.0, -1.0)
correct_predictions = (predicted_labels == hardware_test_labels).sum().item()
accuracy = correct_predictions / num_samples_to_test
f1 = f1_score(
    hardware_test_labels.cpu().numpy(),
    predicted_labels.cpu().numpy(),
    pos_label=1
)

avg_raw_output = predictions.mean().item()
avg_confidence = torch.abs(predictions).mean().item()

print("\n" + "=" * 45)
print("        HARDWARE PERFORMANCE REPORT")
print("=" * 45)
print(f"Circuit Depth:      {stats['Depth']}")
print(f"SWAP Gates:         {stats['SWAPs']}  <-- (Target: 0)")
print_gate_breakdown(stats)
print("-" * 45)
print(f"Samples Tested:     {num_samples_to_test} (Full Test Set)")
print(f"Correctly Guessed:  {correct_predictions} / {num_samples_to_test}")
print(f"Batch Accuracy:     {accuracy:.2%}")
print(f"F1 Score:           {f1:.4f}")
print(f"Avg Raw Output:     {avg_raw_output:.4f}")
print(f"Avg Confidence:     {avg_confidence:.2%}")
print("=" * 45)

## 9. Testing o Estimator

In [ ]:
trained_weights = original_model.quantum_layer.weight.detach().numpy()

X_test_np = X_test_tensor.numpy()
y_test_np = y_test_tensor.numpy().flatten()
num_samples_to_test = len(y_test_np)

transpiled_qc = transpile(original_model.qc, optimization_level=0)
ops_count = transpiled_qc.count_ops()

stats = {
    'Depth': transpiled_qc.depth(),
    'SWAPs': ops_count.get('swap', 0),
}
stats.update(count_gate_types(transpiled_qc))

raw_outputs = original_model.qnn.forward(X_test_np, trained_weights)

raw_outputs = raw_outputs.flatten()

predictions = np.sign(raw_outputs)
predictions[predictions == 0] = 1

correct_predictions = np.sum(predictions == y_test_np)
accuracy = accuracy_score(y_test_np, predictions)
f1 = f1_score(y_test_np, predictions)

avg_raw_output = np.mean(raw_outputs)
avg_confidence = np.mean(np.abs(raw_outputs))

print("\n        HARDWARE PERFORMANCE REPORT")
print("=" * 45)
print(f"Circuit Depth:      {stats['Depth']}")
print(f"SWAP Gates:         {stats['SWAPs']}  <-- (Target: 0)")
print_gate_breakdown(stats)
print("-" * 45)
print(f"Samples Tested:     {num_samples_to_test} (Full Test Set)")
print(f"Correctly Guessed:  {correct_predictions} / {num_samples_to_test}")
print(f"Batch Accuracy:     {accuracy:.2%}")
print(f"F1 Score:           {f1:.4f}")
print(f"Avg Raw Output:     {avg_raw_output:.4f}")
print(f"Avg Confidence:     {avg_confidence:.2%}")

# 4. Ansatz 2: Ring Topology (IQM Spark adapted)



In [ ]:


def ansatz_Odra(n_qubits: int, depth: int) -> QuantumCircuit:
    # Full reverse ring on earlier macro-layers; trim to q0-incident reverse edges only on the last layer.
    n_macro = depth // 2
    theta = ParameterVector("theta", ansatz_trimmed_reverse_q0_param_count(n_qubits, depth))
    qc = QuantumCircuit(n_qubits)
    p = 0

    for j in range(n_macro):
        last_layer = j == n_macro - 1

        for i in range(n_qubits):
            qc.ry(theta[p + i], i)
        p += n_qubits

        for i in range(n_qubits):
            control = i
            target = (i + 1) % n_qubits
            qc.rz(theta[p + i], target)
            qc.cz(control, target)
        p += n_qubits

        for i in range(n_qubits):
            qc.rx(theta[p + i], i)
        p += n_qubits

        if last_layer:
            for k in range(2):
                i = k
                control = i
                target = (i - 1) % n_qubits
                qc.ry(theta[p + k], target)
                qc.cz(control, target)
            p += 2
        else:
            for i in range(n_qubits):
                control = i
                target = (i - 1) % n_qubits
                qc.ry(theta[p + i], target)
                qc.cz(control, target)
            p += n_qubits

    assert p == len(theta)
    return qc

In [ ]:
ma=ansatz_Odra(5,2)
ma.draw(style="mpl")

## 5. Training Configuration and Data Loading

In [ ]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)

# Create DataLoader for batched training
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

## 6. Model Training, Evaluation & Weights upload

We train the hybrid VQC using Mean Squared Error (MSE) loss and the Adam optimizer. The training loop evaluates both training and test performance at each epoch.

### 6.1. Training



In [ ]:
# Initialize model, loss, and optimizer
if 'ansatz_Odra' not in globals():
    def ansatz_Odra(n_qubits: int, depth: int) -> QuantumCircuit:
        # Full reverse ring on earlier macro-layers; trim to q0-incident reverse edges only on the last layer.
        n_macro = depth // 2
        theta = ParameterVector("theta", ansatz_trimmed_reverse_q0_param_count(n_qubits, depth))
        qc = QuantumCircuit(n_qubits)
        p = 0

        for j in range(n_macro):
            last_layer = j == n_macro - 1

            for i in range(n_qubits):
                qc.ry(theta[p + i], i)
            p += n_qubits

            for i in range(n_qubits):
                control = i
                target = (i + 1) % n_qubits
                qc.rz(theta[p + i], target)
                qc.cz(control, target)
            p += n_qubits

            for i in range(n_qubits):
                qc.rx(theta[p + i], i)
            p += n_qubits

            if last_layer:
                for k in range(2):
                    i = k
                    control = i
                    target = (i - 1) % n_qubits
                    qc.ry(theta[p + k], target)
                    qc.cz(control, target)
                p += 2
            else:
                for i in range(n_qubits):
                    control = i
                    target = (i - 1) % n_qubits
                    qc.ry(theta[p + i], target)
                    qc.cz(control, target)
                p += n_qubits

        assert p == len(theta)
        return qc

final_ansatz = ansatz_Odra(NUM_QUBITS, ANSATZ_DEPTH)
odra_model = HybridModel(final_ansatz, NUM_QUBITS)
odra_loss_function = nn.MSELoss()
odra_optimizer = torch.optim.Adam(odra_model.parameters(), lr=LEARNING_RATE)

# Training loop
odra_train_loss_history = []
odra_test_loss_history = []
odra_test_acc_history = []

print(f"Starting training for {EPOCHS} epochs...")
print(f"Odra model: {NUM_QUBITS} qubits, ansatz depth {ANSATZ_DEPTH}")
print(f"Trainable parameters: {sum(p.numel() for p in odra_model.parameters() if p.requires_grad)}")
print("-" * 60)

odra_best_accuracy = 0.0

for epoch in range(EPOCHS):
    train_loss = train_epoch(odra_model, train_loader, odra_optimizer, odra_loss_function)
    test_metrics = evaluate(odra_model, X_test_tensor, y_test_tensor, odra_loss_function)

    current_acc = test_metrics['accuracy']
    if current_acc > odra_best_accuracy:
        odra_best_accuracy = current_acc
        torch.save(odra_model.state_dict(), "quantum_best_Odra_weights_2.pth")
        print(f" New best accuracy: {odra_best_accuracy:.4f} (Odra weights updated)")

    odra_train_loss_history.append(train_loss)
    odra_test_loss_history.append(test_metrics['loss'])
    odra_test_acc_history.append(test_metrics['accuracy'])

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{EPOCHS} | "
              f"Train Loss: {train_loss:.4f} | "
              f"Test Acc: {test_metrics['accuracy']:.4f}")

torch.save(odra_model.state_dict(), "quantum_final_Odra_weights_2.pth")
print("-" * 60)
print(f"Training complete! Best Odra accuracy achieved: {odra_best_accuracy:.4f}")

In [ ]:
with torch.no_grad():
    test_outputs_tensor = odra_model(X_test_tensor)
    test_outputs = test_outputs_tensor.numpy()

predicted = np.where(test_outputs > 0, 1, -1).flatten()

c_matrix_display = ConfusionMatrixDisplay(confusion_matrix=confusion_matrix(y_test, predicted))
c_matrix_display.plot()

epochs = range(1, len(odra_train_loss_history) + 1)

plt.figure(figsize=(14, 5))

# Plot 1: Loss
plt.subplot(1, 2, 1)
plt.plot(epochs, odra_train_loss_history, label='Train Loss', color='blue')
plt.plot(epochs, odra_test_loss_history, label='Test Loss', color='red', linestyle='--')
plt.title('Odra Ansatz - Loss Over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Plot 2: Accuracy
plt.subplot(1, 2, 2)
plt.plot(epochs, odra_test_acc_history, label='Test Accuracy', color='green')
plt.title('Odra Ansatz - Accuracy Over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.show()

print("F1 SCORE: ", f1_score(y_test, predicted), " | ACCURACY SCORE: ", accuracy_score(y_test, predicted), " | BEST ACCURACY SCORE: ", max(odra_test_acc_history))

### 6.3. Weights Upload

To facilitate the immediate verification of our results, this cell provides an option to load the optimal model parameters `quantum_final_Odra_weights.pth`. This allows users to skip the computationally intensive training phase and proceed directly to inference and hardware-level validation using our pre-trained state.

In [ ]:
# Use to load weights from the file, otherwise skip
if 'ansatz_Odra' not in globals():
    def ansatz_Odra(n_qubits: int, depth: int) -> QuantumCircuit:
        # Full reverse ring on earlier macro-layers; trim to q0-incident reverse edges only on the last layer.
        n_macro = depth // 2
        theta = ParameterVector("theta", ansatz_trimmed_reverse_q0_param_count(n_qubits, depth))
        qc = QuantumCircuit(n_qubits)
        p = 0

        for j in range(n_macro):
            last_layer = j == n_macro - 1

            for i in range(n_qubits):
                qc.ry(theta[p + i], i)
            p += n_qubits

            for i in range(n_qubits):
                control = i
                target = (i + 1) % n_qubits
                qc.rz(theta[p + i], target)
                qc.cz(control, target)
            p += n_qubits

            for i in range(n_qubits):
                qc.rx(theta[p + i], i)
            p += n_qubits

            if last_layer:
                for k in range(2):
                    i = k
                    control = i
                    target = (i - 1) % n_qubits
                    qc.ry(theta[p + k], target)
                    qc.cz(control, target)
                p += 2
            else:
                for i in range(n_qubits):
                    control = i
                    target = (i - 1) % n_qubits
                    qc.ry(theta[p + i], target)
                    qc.cz(control, target)
                p += n_qubits

        assert p == len(theta)
        return qc

final_ansatz = ansatz_Odra(NUM_QUBITS, ANSATZ_DEPTH)
odra_model = HybridModel(final_ansatz, NUM_QUBITS)
FILE_PATH = "quantum_final_Odra_weights_2.pth"
try:
    odra_model.load_state_dict(torch.load(FILE_PATH))
    odra_model.eval()
    sanity_metrics = evaluate(odra_model, X_test_tensor, y_test_tensor, nn.MSELoss())
    print(f"Weights loaded successfully from {FILE_PATH}! You can now proceed to hardware testing.")
    print(f"Estimator sanity accuracy: {sanity_metrics['accuracy']:.4f}")
except FileNotFoundError:
    print("Error: The weights file was not found.")

### 7.2. Constructing the Hardware-Ready Quantum Neural Network (QNN)

This section builds the bridge between PyTorch machine learning environment and the physical IQM quantum hardware. We construct a new quantum neural network explicitly bound to the hardware estimator, and then transfer our pre-trained weights into it for real-world inference.



**Key Steps in this Cell:**
* **Hardware Estimator Setup:** Initializes the custom `IQMBackendEstimator` to handle execution and shot-counting on the physical backend.
* **Circuit Assembly:** Constructs the physical quantum blueprint (`hw_qc`) by combining our data-encoding Feature Map and our trainable Ansatz.
* **Defining the Observable:** Sets up a `SparsePauliOp` (`IIIIZ`) to measure the Pauli-Z expectation value of Qubit 0, aligning with our estimator's logic.
* **PyTorch Integration (`TorchConnector`):** Wraps the Qiskit `EstimatorQNN` into a PyTorch module (`iqm_model`), allowing it to natively accept PyTorch tensors as input.
* **Weight Transfer & Evaluation Mode:** Extracts the optimized weights from the classically-trained model and injects them into our new hardware-linked model. Calling `.eval()` locks the model into testing mode, ensuring we don't waste hardware resources calculating gradients.

In [ ]:
# Connect to IQM
# ---------------------------------------------------------
try:
    # Replace URL/Token as needed
    provider = IQMProvider("https://odra5.e-science.pl/", token=input("Enter IQM Token: "))
    iqm_backend = provider.get_backend()
    print(f"Connected to backend: {iqm_backend.name}")
except Exception as e:
    print(f"Connection error: {e}")
    # Stop execution if connection fails (optional safety)
    exit()

# Instantiate the Bridge & QNN
# ---------------------------------------------------------

# Create the hardware estimator
ODRA_SHOTS = 512
SHOTS = 512
MAX_CIRCUITS_PER_JOB = 275

odra_hardware_estimator = IQMBackendEstimator(
    iqm_backend,
    options={
        "shots": SHOTS,
        "max_circuits_per_job": MAX_CIRCUITS_PER_JOB,
    }
)

print("Building Odra Hardware QNN...")

# Build Circuit Components
odra_hw_ansatz = ansatz_Odra(NUM_QUBITS, ANSATZ_DEPTH)
odra_hw_feature_map = odra_model._create_angle_encoding(NUM_QUBITS)

odra_hw_qc = QuantumCircuit(NUM_QUBITS)
odra_hw_qc.compose(odra_hw_feature_map, qubits=range(NUM_QUBITS), inplace=True)
odra_hw_qc.compose(odra_hw_ansatz, inplace=True)

# Observable
observable = SparsePauliOp.from_list([("I" * (NUM_QUBITS - 1) + "Z", 1)])

# Create QNN with the HARDWARE ESTIMATOR
odra_hw_qnn = EstimatorQNN(
    circuit=odra_hw_qc,
    observables=observable,
    input_params=list(odra_hw_feature_map.parameters),
    weight_params=list(odra_hw_ansatz.parameters),
    estimator=odra_hardware_estimator
)

# Create Torch Layer
odra_iqm_model = TorchConnector(odra_hw_qnn)

# Load trained weights
# ---------------------------------------------------------
print("Loading Odra weights...")
try:
    odra_iqm_model.load_state_dict(odra_model.quantum_layer.state_dict())
    odra_iqm_model.eval()
    print("Odra weights transferred to IQM model successfully!")
except RuntimeError as e:
    print("\n[!] Weight Loading Error: The shape of the trained weights does not match the hardware model.")
    print("Ensure the model you are loading from was trained with N_QUBITS=5 and the exact same Odra ansatz.")
    print(f"Details: {e}")

## 8. Testing Single Input on IQM Spark

This section sends a randomly chosen sample to IQM Spark to test our model on real quantum hardware.

In [ ]:
qnn_circuit = odra_iqm_model.neural_network.circuit
transpiled_qnn_circuit, stats = get_circuit_stats(qnn_circuit, iqm_backend)

hardware_test_inputs = X_test_tensor
hardware_test_labels = y_test_tensor.view(-1)
num_samples_to_test = len(hardware_test_inputs)

circuits_per_job = odra_hardware_estimator._options.get("max_circuits_per_job") or num_samples_to_test
num_jobs = int(np.ceil(num_samples_to_test / circuits_per_job))

print(f"Loaded ALL {num_samples_to_test} test samples for hardware testing.")
print(f"\nSending {num_jobs} job(s) containing up to {circuits_per_job} batched circuits to IQM Spark...")
print("\nTranspiled circuit used on the quantum computer:")
display(transpiled_qnn_circuit.draw(style="mpl", idle_wires=False))

with torch.no_grad():
    predictions = odra_iqm_model(hardware_test_inputs).view(-1)

predicted_labels = torch.where(predictions > 0, 1.0, -1.0)
correct_predictions = (predicted_labels == hardware_test_labels).sum().item()
accuracy = correct_predictions / num_samples_to_test
f1 = f1_score(
    hardware_test_labels.cpu().numpy(),
    predicted_labels.cpu().numpy(),
    pos_label=1
)
avg_raw_output = predictions.mean().item()
avg_confidence = torch.abs(predictions).mean().item()

print("\n" + "=" * 45)
print("        HARDWARE PERFORMANCE REPORT")
print("=" * 45)
print(f"Circuit Depth:      {stats['Depth']}")
print(f"SWAP Gates:         {stats['SWAPs']}  <-- (Target: 0)")
print_gate_breakdown(stats)
print("-" * 45)
print(f"Samples Tested:     {num_samples_to_test} (Full Test Set)")
print(f"Correctly Guessed:  {correct_predictions} / {num_samples_to_test}")
print(f"Batch Accuracy:     {accuracy:.2%}")
print(f"F1 Score:           {f1:.4f}")
print(f"Avg Raw Output:     {avg_raw_output:.4f}")
print(f"Avg Confidence:     {avg_confidence:.2%}")
print("=" * 45)

## 9. Testing Odra_ansatz on Estimator

In [ ]:
trained_weights = odra_model.quantum_layer.weight.detach().numpy()

X_test_np = X_test_tensor.numpy()
y_test_np = y_test_tensor.numpy().flatten()
num_samples_to_test = len(y_test_np)

transpiled_qc = transpile(odra_model.qc, optimization_level=0)
ops_count = transpiled_qc.count_ops()

stats = {
    'Depth': transpiled_qc.depth(),
    'SWAPs': ops_count.get('swap', 0),
}
stats.update(count_gate_types(transpiled_qc))

raw_outputs = odra_model.qnn.forward(X_test_np, trained_weights)

raw_outputs = raw_outputs.flatten()

predictions = np.sign(raw_outputs)
predictions[predictions == 0] = 1

correct_predictions = np.sum(predictions == y_test_np)
accuracy = accuracy_score(y_test_np, predictions)
f1 = f1_score(y_test_np, predictions)

avg_raw_output = np.mean(raw_outputs)
avg_confidence = np.mean(np.abs(raw_outputs))

print("\n        HARDWARE PERFORMANCE REPORT")
print("=" * 45)
print(f"Circuit Depth:      {stats['Depth']}")
print(f"SWAP Gates:         {stats['SWAPs']}  <-- (Target: 0)")
print_gate_breakdown(stats)
print("-" * 45)
print(f"Samples Tested:     {num_samples_to_test} (Full Test Set)")
print(f"Correctly Guessed:  {correct_predictions} / {num_samples_to_test}")
print(f"Batch Accuracy:     {accuracy:.2%}")
print(f"F1 Score:           {f1:.4f}")
print(f"Avg Raw Output:     {avg_raw_output:.4f}")
print(f"Avg Confidence:     {avg_confidence:.2%}")